# 🧪 pKa Prediction Pipeline with XGBoost Feature Selection and QLattice Grid Search

In [1]:
# --- 1. Imports ---
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Fragments
from rdkit.Chem import SaltRemover
from mordred import Calculator, descriptors
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
import feyn
from tqdm import tqdm
import logging
from func_timeout import func_timeout, FunctionTimedOut

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

In [2]:
# --- 2. Load OP2 Dataset ---
train_file = './dw_data/Opt2_acidic_tr.csv'
test_file = './dw_data/Opt2_acidic_tst.csv'
target = 'pKa'
smiles_col = 'OriginalSmiles'

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)
for df in [train_df, test_df]:
    df[target] = pd.to_numeric(df[target], errors='coerce')

In [3]:
# --- 3. Salt Removal + SMILES to Mol ---
saltRemover = SaltRemover.SaltRemover(defnFilename='./Salts.txt')
for df in [train_df, test_df]:
    df['Mol'] = df[smiles_col].astype(str).apply(lambda s: saltRemover.StripMol(Chem.MolFromSmiles(s)))

In [4]:
# --- Descriptor Computation (Fahad's Original Style) ---
calc = Calculator(descriptors, ignore_3D=True)

def safe_call(func, mol, timeout=1):
    try:
        return func_timeout(timeout, func, args=(mol,))
    except (FunctionTimedOut, Exception) as e:
        return np.nan

def compute_rdkit_descriptors(mol):
    descriptor_funcs = {name: func for name, func in Descriptors.descList}
    if mol is None or Chem.MolToSmiles(mol) == '':
        return None
    return {name: safe_call(func, mol, timeout=1) for name, func in descriptor_funcs.items()}

def compute_fragment_counts(mol):
    frag_funcs = {name: func for name, func in Fragments.__dict__.items() if callable(func) and name.startswith('fr_')}
    if mol is None or Chem.MolToSmiles(mol) == '':
        return None
    return {name: safe_call(func, mol, timeout=1) for name, func in frag_funcs.items()}

def compute_descriptors_for_df(df):
    mols = df['Mol'].tolist()
    rdkit_list, frag_list = [], []
    from tqdm import tqdm
    for mol in tqdm(mols, total=len(mols), desc='Computing Descriptors'):
        rdkit_desc = compute_rdkit_descriptors(mol)
        frag_desc = compute_fragment_counts(mol)
        rdkit_list.append(rdkit_desc if rdkit_desc is not None else {})
        frag_list.append(frag_desc if frag_desc is not None else {})
    rdkit_df = pd.DataFrame(rdkit_list)
    frag_df = pd.DataFrame(frag_list)
    mordred_df = calc.pandas(mols)
    full_desc_df = pd.concat([rdkit_df, frag_df, mordred_df], axis=1)
    non_zero_std_cols = full_desc_df.std(numeric_only=True)
    full_desc_df = full_desc_df[non_zero_std_cols[non_zero_std_cols > 0].index]
    full_desc_df = full_desc_df.apply(pd.to_numeric, errors='coerce')
    full_df = pd.concat([df[[target]].reset_index(drop=True), full_desc_df.reset_index(drop=True)], axis=1)
    return full_df.dropna()


In [5]:
# --- 5. Generate Descriptors ---
train_data = compute_descriptors_for_df(train_df)
test_data = compute_descriptors_for_df(test_df)

  3%|██                                                                              | 61/2321 [00:01<00:24, 93.92it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Autocorrelation.py:97: RuntimeWarning: Mean of empty slice.
  return avec - avec.mean()
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Constitutional.py:80: RuntimeWarning: invalid value encountered in double_scalars
  return S / self.mol.GetNumAtoms()


 37%|█████████████████████████████▍                                                 | 288/774 [00:03<00:04, 100.28it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Autocorrelation.py:97: RuntimeWarning: Mean of empty slice.
  return avec - avec.mean()
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\mordred\Constitutional.py:80: RuntimeWarning: invalid value encountered in double_scalars
  return S / self.mol.GetNumAtoms()


100%|████████████████████████████████████████████████████████████████████████████████| 774/774 [00:09<00:00, 77.86it/s]


In [6]:
# Define the sanitize function within this cell
def sanitize(df):
    df = df.loc[:, df.columns.notna()]      # Drop unnamed/NaN columns
    df = df.loc[:, df.notna().all()]          # Drop columns with any NaNs
    df.columns = [str(col) for col in df.columns]  # Convert all columns to strings
    df = df.loc[:, ~df.columns.duplicated()]  # Remove duplicate columns
    return df.reset_index(drop=True)

# Sanitize the descriptor datasets
print("Sanitizing training and test datasets...")
train_data = sanitize(train_data)
test_data = sanitize(test_data)
print("✅ Column cleanup complete.")

# Align training and test features
X_train = train_data.drop(columns=[target])
X_test = test_data.drop(columns=[target])

# Add missing columns to X_test and remove extra columns so that X_test matches X_train
missing_cols = set(X_train.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0  # You may choose another default value if needed

extra_cols = set(X_test.columns) - set(X_train.columns)
if extra_cols:
    X_test = X_test.drop(columns=list(extra_cols))

# Ensure the same column order
X_test = X_test[X_train.columns]

y_train = train_data[target]
y_test = test_data[target]

print("X_train shape:", X_train.shape, "X_test shape:", X_test.shape)

# Train the XGBoost model and print training progress
print("Training XGBoost model...")
from xgboost import XGBRegressor
xgb_model = XGBRegressor(
    colsample_bytree=0.8,
    learning_rate=0.1,
    max_depth=6,
    n_estimators=300,
    reg_lambda=1,
    subsample=0.8,
    objective='reg:squarederror',
    random_state=42
)
xgb_model.fit(X_train, y_train)
print("✅ XGBoost training complete.")

# Calculate and print metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("Evaluating model performance...")
train_preds = xgb_model.predict(X_train)
train_r2 = r2_score(y_train, train_preds)
train_mse = mean_squared_error(y_train, train_preds)
train_mae = mean_absolute_error(y_train, train_preds)
print("Training Metrics: R2: {:.4f}, MSE: {:.4f}, MAE: {:.4f}".format(train_r2, train_mse, train_mae))

test_preds = xgb_model.predict(X_test)
test_r2 = r2_score(y_test, test_preds)
test_mse = mean_squared_error(y_test, test_preds)
test_mae = mean_absolute_error(y_test, test_preds)
print("Test Metrics: R2: {:.4f}, MSE: {:.4f}, MAE: {:.4f}".format(test_r2, test_mse, test_mae))

# Compute feature importances from the XGBoost model
importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 10 Features:\n", importances.head(10))


Sanitizing training and test datasets...
✅ Column cleanup complete.
X_train shape: (2293, 717) X_test shape: (765, 717)
Training XGBoost model...
✅ XGBoost training complete.
Evaluating model performance...
Training Metrics: R2: 0.9997, MSE: 0.0040, MAE: 0.0484
Test Metrics: R2: 0.8312, MSE: 1.9402, MAE: 0.8868
Top 10 Features:
 nAcid               0.251568
fr_tetrazole        0.055906
fr_COO              0.055835
SdsssP              0.042854
SdssS               0.018750
fr_nitrile          0.018443
fr_SH               0.017629
MaxPartialCharge    0.017503
fr_quatN            0.015526
nBase               0.014286
dtype: float32


In [ ]:
# --- QLattice Grid Search with Progress Bar and Success Prints ---

from itertools import product
import tqdm

# Select top features based on importances from XGBoost
N = 150
top_features = importances.head(N).index.tolist()
train_selected = train_data[[target] + top_features]
test_selected = test_data[[target] + top_features]
ql = feyn.QLattice()

# Define the parameter grid and build all parameter combinations
param_grid = {
    'n_epochs': [100, 200],
    'max_complexity': [200],
    'criterion': ['aic', 'bic']
}
param_combinations = list(product(param_grid['n_epochs'], param_grid['max_complexity'], param_grid['criterion']))
print("Starting QLattice grid search on {} combinations...".format(len(param_combinations)))

results = []
for epochs, complexity, crit in tqdm.tqdm(param_combinations, desc="QLattice grid search"):
    models = ql.auto_run(train_selected, output_name=target, n_epochs=epochs,
                           threads=16, max_complexity=complexity, criterion=crit)
    if models:
        try:
            predictions = models[0].predict(test_selected)
            r2 = r2_score(test_selected[target], predictions)
        except Exception as e:
            r2 = np.nan
    else:
        r2 = np.nan
    results.append({'epochs': epochs, 'complexity': complexity, 'criterion': crit, 'r2': r2})

print("✅ QLattice grid search complete.")
results_df = pd.DataFrame(results).sort_values(by='r2', ascending=False)
print("Top QLattice results:")
print(results_df.head())


In [ ]:
# --- 9. Export Computed Descriptor Datasets ---
train_data.to_csv('train_descriptors_op2.csv', index=False)
test_data.to_csv('test_descriptors_op2.csv', index=False)
print('✅ Descriptors saved to CSV.')